prothomaakter937@gmail.com


# Week 01 Assignment  
## Data Quality, Evaluation, Scaling, and Encoding

**Student name: Prothoma AKter**   

This is a small assignment that connects topics from Module 1, 2, and 3.  
You must complete it in this Colab notebook.

You will need to use concepts that appeared in the videos:
- Module 1 and 2: basic descriptive statistics, proportions, confusion matrix, accuracy, precision, recall
- Module 3: standardization, min max scaling, nominal vs ordinal, one hot encoding, ordinal encoding, Euclidean and Manhattan distance

Please do not use any extra libraries beyond `pandas`, `numpy`.



---
## 0. Setup and Dataset

We will use a dataset that should have columns given below:

- `user_id`  
- `age`  
- `monthly_income` (numeric)  
- `daily_screen_time_min` (numeric)  
- `daily_app_opens` (numeric)  
- `true_label` and `pred_label` for a binary classification task (0 or 1)  
- `satisfaction_level` (for example: `Low`, `Medium`, `High`)  
- `city_type` (for example: `Urban`, `Suburban`, `Rural`)


In [288]:
# Cell 1: Imports
import pandas as pd
import numpy as np

In [289]:
# Cell 2: Load the dataset (Already done for you)
df = pd.read_csv("https://drive.google.com/uc?export=download&id=1OmDDCh4MD1TtvAemnwVDyz5zwCIXJ220")

# Show first few rows
df.head()

,user_id,age,monthly_income,daily_screen_time_min,daily_app_opens,true_label,pred_label,satisfaction_level,city_type
0,1,43,3734.19,109,48,0,0,Medium,Suburban
1,2,49,2594.19,194,7,0,0,Low,Urban
2,3,19,3550.47,146,36,1,0,High,Rural
3,4,19,3821.18,287,14,1,0,High,Suburban
4,5,63,1750.84,66,46,0,0,Medium,Suburban



### 0.1 Check your dataset

1. Confirm that the dataset loaded correctly.  
2. Check that you have at least these columns:  
   - numeric: `age`, `monthly_income`, `daily_screen_time_min`, `daily_app_opens`  
   - labels: `true_label`, `pred_label`  
   - categorical: `satisfaction_level`, `city_type`  



---
## Part A - Module 1 and 2 Review

In this part you will do simple descriptive statistics and basic classification evaluation.



### Q1. Descriptive statistics on a numeric feature

Choose one numeric column, for example `daily_screen_time_min`.


In [290]:
# Q1.1: Choose your numeric column here [We already write this ans]
num_col = "daily_screen_time_min"

df[num_col].describe()

,daily_screen_time_min
count,100.000000
mean,181.890000
std,68.886951
min,60.000000
25%,122.000000
50%,178.000000
75%,243.750000
max,299.000000



> **Q1.2 Short answer: [Marks: 05]**  
> Look at the count, mean, min, max, and standard deviation for your chosen column.  
> In 2 to 3 sentences, comment on what you see.  
> For example, does the max look very far from the mean, or does it look quite close?

Write your answer here:

>  The statistical summary shows that the mean value of 181.89 is quite close to the median of 178, indicating a fairly balanced distribution. The data spreads evenly between a minimum of 60 and a maximum of 299 with a standard deviation of about 68.89. Given this spread, the maximum value does not appear unusually extreme or far from the mean, suggesting the absence of significant outliers.
>  
>  



### Q2. Proportion of positive class

Use the `true_label` column, where 1 means "positive" and 0 means "negative".


In [291]:
# Q2.1: Compute proportion of positive class [We already write this ans]
label_col = "true_label"

positive_count = (df[label_col] == 1).sum()
total_count = df.shape[0]
positive_proportion = positive_count / total_count

print("Positive count:", positive_count)
print("Total samples:", total_count)
print("Proportion of positive class:", positive_proportion)

Positive count: 52
Total samples: 100
Proportion of positive class: 0.52



> **Q2.2 Short answer: [5 marks]**  
> In 1 to 2 sentences, explain what this proportion tells you about your dataset.  
> For example, is the dataset balanced between 0 and 1, or is one class much more common?

Write your answer here:

>> 52 out of 100 samples turned out positive, so the proportion is 0.52 basically a 50-50 split. Since neither class outnumbers the other by much, I'd say this dataset is fairly balanced between positive and negative labels.
>  
>  



### Q3. Confusion matrix and basic metrics

For this question, use:
- `true_label` as the actual label  
- `pred_label` as the model prediction


In [292]:
# Q3.1: Manually compute TP, TN, FP, FN [We already write this ans]
true_col = "true_label"
pred_col = "pred_label"

tp = ((df[true_col] == 1) & (df[pred_col] == 1)).sum()
tn = ((df[true_col] == 0) & (df[pred_col] == 0)).sum()
fp = ((df[true_col] == 0) & (df[pred_col] == 1)).sum()
fn = ((df[true_col] == 1) & (df[pred_col] == 0)).sum()

print("TP:", tp)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)

TP: 28
TN: 27
FP: 21
FN: 24


In [293]:
# Q3.2: Compute accuracy, precision, recall [We already write this ans]
accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

Accuracy: 0.55
Precision: 0.5714285714285714
Recall: 0.5384615384615384



> **Q3.3 Short answer: [10 marks]**  
> In 3 to 4 sentences, briefly comment on the model using these three metrics.  
> For example, is the model catching most positives (high recall) or being careful when it predicts positive (high precision)?

Write your answer here:

> The model's accuracy comes out to 0.55, so it's only getting the label right a little over half the time not great. Precision is around 0.57, meaning when it does predict positive, it's correct about 57% of the time, and recall is close to 0.54, so it's catching roughly 54% of the actual positive cases. Since precision and recall are both close together and neither one is particularly high, it's clear the model's overall performance is fairly weak and could use some improvement.
>
>
>  
>  



---
## Part B - Module 3: Scaling and Encoding

Now we will pick a few features and apply scaling and encoding.



### Q4. Standardization and Min max scaling

Use one numeric column, `monthly_income`.


In [294]:
# Q4.1: Choose the numeric column [2 marks]
numerical_col =df["monthly_income"]
numerical_col.head()

,monthly_income
0,3734.19
1,2594.19
2,3550.47
3,3821.18
4,1750.84


In [295]:
# Q4.2: Standardization with z-score [10 marks]
mean=numerical_col.mean()
std=numerical_col.std()
z_score=(numerical_col-mean)/std
z_score=z_score.round(2)
z_score.head()

,monthly_income
0,0.94
1,-0.32
2,0.74
3,1.04
4,-1.26


In [296]:
# Q4.3: Min max scaling implementation [10 marks]
min_v=numerical_col.min()
max_v=numerical_col.max()
min_max_scaled=(numerical_col-min_v)/(max_v-min_v)
min_max_scaled=min_max_scaled.round(2)
min_max_scaled

,monthly_income
0,0.68
1,0.39
2,0.63
3,0.70
4,0.19
...,...
95,0.73
96,0.68
97,0.33
98,0.50



> **Q4.4 Short answer: [3 marks]**  
> Compare the standardized and min max scaled columns in 2 to 3 sentences.  
> Mention what kind of range each one uses and how the numbers look.

Write your answer here:

>  Standardization transforms the data so it has a mean of 0 and a standard deviation of 1, resulting in a mix of positive and negative numbers. On the other hand, min-max scaling squishes all the values into a strict range between 0 and 1. Even though they look different, both methods keep the original shape of the data intact while shifting them to completely new scales.
>  
>  



### Q5. One hot and ordinal encoding

We will use:
- `city_type` as a nominal feature  
- `satisfaction_level` as an ordinal feature with order `Low` < `Medium` < `High`  


In [297]:
# Q5.1: One hot encoding for city_type using pandas [10 marks]
df_encoded=pd.get_dummies(df["city_type"],prefix='city_type',dtype=int)
df_encoded

,city_type_Rural,city_type_Suburban,city_type_Urban
0,0,1,0
1,0,0,1
2,1,0,0
3,0,1,0
4,0,1,0
...,...,...,...
95,0,0,1
96,0,0,1
97,1,0,0
98,0,0,1


In [298]:
# Q5.2: Attach one hot encoded columns to df [5 marks]
df=df.drop('city_type',axis=1)
df=pd.concat([df,df_encoded],axis=1)
df

,user_id,age,monthly_income,daily_screen_time_min,daily_app_opens,true_label,pred_label,satisfaction_level,city_type_Rural,city_type_Suburban,city_type_Urban
0,1,43,3734.19,109,48,0,0,Medium,0,1,0
1,2,49,2594.19,194,7,0,0,Low,0,0,1
2,3,19,3550.47,146,36,1,0,High,1,0,0
3,4,19,3821.18,287,14,1,0,High,0,1,0
4,5,63,1750.84,66,46,0,0,Medium,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
95,96,20,3975.90,259,15,0,1,Medium,0,0,1
96,97,52,3764.77,295,16,1,0,Low,0,0,1
97,98,35,2336.44,108,46,0,1,High,1,0,0
98,99,18,3007.53,202,42,1,1,High,0,0,1


In [299]:
# Q5.3: Ordinal encoding for satisfaction_level [10 marks]
ordinal_encoding={'Medium':2, 'Low':1, 'High':3}
df['satisfaction_level']=df['satisfaction_level'].map(ordinal_encoding).astype(int)
df

,user_id,age,monthly_income,daily_screen_time_min,daily_app_opens,true_label,pred_label,satisfaction_level,city_type_Rural,city_type_Suburban,city_type_Urban
0,1,43,3734.19,109,48,0,0,2,0,1,0
1,2,49,2594.19,194,7,0,0,1,0,0,1
2,3,19,3550.47,146,36,1,0,3,1,0,0
3,4,19,3821.18,287,14,1,0,3,0,1,0
4,5,63,1750.84,66,46,0,0,2,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
95,96,20,3975.90,259,15,0,1,2,0,0,1
96,97,52,3764.77,295,16,1,0,1,0,0,1
97,98,35,2336.44,108,46,0,1,3,1,0,0
98,99,18,3007.53,202,42,1,1,3,0,0,1



> **Q5.4 Short answer: [5 marks]**  
> In 2 to 3 sentences, explain why one hot encoding is suitable for `city_type`  
> and why ordinal encoding is suitable for `satisfaction_level`.

Write your answer here:

> One-hot encoding makes sense for city_type since it's a nominal feature categories like Urban or Suburban don't have any natural order between them. Satisfaction_level, on the other hand, works better with ordinal encoding because Low, Medium, and High actually follow a clear ranking, so preserving that order matters.
>  
>  



---
## Part C - Module 3: Distances between users

For this small part we will work with vectors based on scaled numeric features.



### Q6. Euclidean and Manhattan distance

Build 2D vectors for user 0 and user 1 using:
- `income_std`  
- `daily_app_opens` (or its min max scaled version if you prefer)


In [300]:
# Q6.1: Build 2D vectors for first two users [We already write this ans]
vec_cols = ["monthly_income", "daily_app_opens"]

v1 = df.loc[0, vec_cols].values
v2 = df.loc[1, vec_cols].values

print("v1:", v1)
print("v2:", v2)

v1: [3734.19   48.  ]
v2: [2594.19    7.  ]


In [301]:
# Q6.2: Euclidean distance computation [5 marks]
euclidean_distance =np.linalg.norm(v1-v2)
print("Euclidean distance:", euclidean_distance)

Euclidean distance: 1140.7370424422975


In [302]:
# Q6.3: Manhattan distance computation [5 marks]
manhattan_distance=np.linalg.norm(v1-v2,ord=1)
print("Manhattan distance:", manhattan_distance)

Manhattan distance: 1181.0



> **Q6.4 Short answer: [5 marks]**  
> Which one is larger in your result, Euclidean or Manhattan distance  
> and why does that usually happen based on their formulas?

Write your answer here:

>  In my result, Manhattan distance came out larger than Euclidean distance. That's because Manhattan distance just adds up the absolute differences along each axis, while Euclidean takes the direct straight-line path between the two points. Since moving along grid lines is always going to be equal to or longer than cutting straight across diagonally, Manhattan distance ends up being greater than or equal to Euclidean in general.
>  
>  



---
## Final Reflection [10 marks]

> In 4 to 6 sentences, describe how the three modules connect in this assignment.  
> Mention:
> - One idea from Module 1 or 2 that you used  
> - One idea from Module 3 that you used  
> - How these ideas together help you understand a dataset more deeply

Write your reflection here:

>This assignment connects evaluation, encoding, and distance metrics into one workflow. From Module 1/2, I used accuracy, precision, and recall to evaluate predictions against true labels. From Module 3, I used one-hot and ordinal encoding to convert categorical features into numbers, then applied Euclidean and Manhattan distance to compare users. Together, these ideas helped me both measure model performance and represent the data numerically for deeper analysis.
>  
>  



## End of Assignment

Before submitting:
- Run all cells from top to bottom.  
- Check that all answer sections are filled.  
- Instruction video অনুযায়ী আমাদের দেয়া Colab ফাইলটি থেকে প্রথম একটি Save copy in drive করে নিবা। এরপর Google colab এর মধ্যে কোডগুলো করবে এবং সেই ফাইলটি ‘Anyone with the link’ & ‘View’ Access দিয়ে ফাইলটির Shareble Link টি সাবমিট করবে।
